# Cabardo Studio AI Generator — OmniLocal 1.2 Free Colab T4 16 GB Backend

**Real Open-Weights AI Video + Synchronized Audio ($0 Dedicated NVIDIA T4 16 GB · Zero Queue · Zero Watermarks):**
- **`Lightricks/LTX-Video-0.9.7-distilled`**: Fast 8-step real AI video (**Text-to-Video**, **Image-to-Video**, and **First→Last Frame Interpolation**)
- **`Wan-AI/Wan2.1-T2V-1.3B-Diffusers`** & **`THUDM/CogVideoX-2b`**
- **`cvssp/audioldm2` + FFmpeg AAC Muxing**: Automatic synchronized foley & cinema soundscape

**How to run (2 clicks):**
1. Confirm top-right says **T4** (if not: `Runtime -> Change runtime type -> T4 GPU -> Save`).
2. Click **Runtime -> Run all** (~45s setup), then paste the printed `https://*.trycloudflare.com` URL into **https://flow-studio-pwa.vercel.app -> Engine Settings -> Test Connection -> Save**.


In [ ]:
# @title Step 1: Verify Free NVIDIA T4 16 GB GPU & Install OmniLocal Dependencies (~40s)
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU attached! Click Runtime -> Change runtime type -> T4 GPU -> Save, then click Run All."
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"GPU Ready: {gpu_name} ({vram_gb:.1f} GB VRAM)")
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
!pip -q install -U "diffusers>=0.33.0" transformers accelerate bitsandbytes safetensors sentencepiece protobuf ftfy "imageio[ffmpeg]" pillow numpy scipy fastapi uvicorn
print("All OmniLocal 1.2 Colab T4 dependencies installed cleanly.")


In [ ]:
# @title Step 2: Launch OmniLocal 1.2 T4 Video + Synced Audio Server (FastAPI + Free Cloudflare Tunnel)
import asyncio
import base64
import gc
import io
import os
import re
import shutil
import subprocess
import threading
import time
import urllib.request
import uuid
import wave
from typing import Optional

import numpy as np
import torch
from PIL import Image, ImageOps
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from pydantic import BaseModel
import uvicorn

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU attached! Click Runtime -> Change runtime type -> T4 GPU -> Save, then Run All again."
    )

os.makedirs("outputs", exist_ok=True)
COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

app = FastAPI(title="Cabardo Studio OmniLocal 1.2 Colab T4 Backend")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
)

GPU_LOCK = threading.Lock()
STATE = {
    "active_pipe_key": None,
    "pipe": None,
    "jobs": {},
}

def _unload_pipe():
    if STATE["pipe"] is not None:
        del STATE["pipe"]
        STATE["pipe"] = None
        STATE["active_pipe_key"] = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def _decode_pil(data_url: str, width: int, height: int) -> Image.Image:
    raw_b64 = data_url.split(",", 1)[1]
    img = Image.open(io.BytesIO(base64.b64decode(raw_b64))).convert("RGB")
    return ImageOps.fit(img, (width, height), method=Image.Resampling.LANCZOS)

def _load_image_pipe(model_key: str):
    target_key = f"img:{model_key}"
    if STATE["active_pipe_key"] == target_key and STATE["pipe"] is not None:
        return STATE["pipe"]
    _unload_pipe()
    from diffusers import AutoPipelineForText2Image
    if model_key == "sdxl-turbo":
        pipe = AutoPipelineForText2Image.from_pretrained(
            "stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16"
        ).to("cuda")
    else:
        pipe = AutoPipelineForText2Image.from_pretrained(
            "stabilityai/stable-diffusion-xl-base-1.0",
            torch_dtype=torch.float16,
            variant="fp16",
            use_safetensors=True,
        )
        pipe.enable_model_cpu_offload()
        pipe.enable_vae_slicing()
        pipe.enable_vae_tiling()
    STATE["pipe"] = pipe
    STATE["active_pipe_key"] = target_key
    return pipe

def _load_video_pipe(model_key: str, mode_variant: str):
    target_key = f"vid:{model_key}:{mode_variant}"
    if STATE["active_pipe_key"] == target_key and STATE["pipe"] is not None:
        return STATE["pipe"]
    _unload_pipe()
    from diffusers.quantizers import PipelineQuantizationConfig
    bnb_kwargs = {
        "load_in_4bit": True,
        "bnb_4bit_quant_type": "nf4",
        "bnb_4bit_compute_dtype": COMPUTE_DTYPE,
    }
    if model_key in ("ltx-0.9.7-distilled", "ltx-video"):
        is_13b = model_key == "ltx-0.9.7-distilled"
        repo = "Lightricks/LTX-Video-0.9.7-distilled" if is_13b else "Lightricks/LTX-Video-0.9.5"
        qcfg = PipelineQuantizationConfig(
            quant_backend="bitsandbytes_4bit",
            quant_kwargs=bnb_kwargs,
            components_to_quantize=["text_encoder", "transformer"] if is_13b else ["text_encoder"],
        )
        if mode_variant == "flf":
            from diffusers import LTXConditionPipeline
            pipe = LTXConditionPipeline.from_pretrained(repo, quantization_config=qcfg, torch_dtype=COMPUTE_DTYPE, low_cpu_mem_usage=True)
        elif mode_variant == "i2v":
            from diffusers import LTXImageToVideoPipeline
            pipe = LTXImageToVideoPipeline.from_pretrained(repo, quantization_config=qcfg, torch_dtype=COMPUTE_DTYPE, low_cpu_mem_usage=True)
        else:
            from diffusers import LTXPipeline
            pipe = LTXPipeline.from_pretrained(repo, quantization_config=qcfg, torch_dtype=COMPUTE_DTYPE, low_cpu_mem_usage=True)
        try:
            pipe.enable_model_cpu_offload()
        except Exception:
            pipe.to("cuda")
        pipe.vae.enable_tiling()
    elif model_key == "wan2.1-1.3b":
        from diffusers import WanPipeline
        qcfg = PipelineQuantizationConfig(
            quant_backend="bitsandbytes_4bit",
            quant_kwargs=bnb_kwargs,
            components_to_quantize=["text_encoder"],
        )
        pipe = WanPipeline.from_pretrained("Wan-AI/Wan2.1-T2V-1.3B-Diffusers", quantization_config=qcfg, torch_dtype=COMPUTE_DTYPE, low_cpu_mem_usage=True)
        try:
            pipe.enable_model_cpu_offload()
        except Exception:
            pipe.to("cuda")
    else:
        from diffusers import CogVideoXPipeline
        qcfg = PipelineQuantizationConfig(
            quant_backend="bitsandbytes_4bit",
            quant_kwargs=bnb_kwargs,
            components_to_quantize=["text_encoder"],
        )
        pipe = CogVideoXPipeline.from_pretrained("THUDM/CogVideoX-2b", quantization_config=qcfg, torch_dtype=torch.float16, low_cpu_mem_usage=True)
        try:
            pipe.enable_model_cpu_offload()
        except Exception:
            pipe.to("cuda")
        pipe.vae.enable_slicing()
        pipe.vae.enable_tiling()
    STATE["pipe"] = pipe
    STATE["active_pipe_key"] = target_key
    return pipe

def _synthesize_and_mux_audio(video_path: str, prompt: str, duration_sec: float, seed: int) -> bool:
    if not shutil.which("ffmpeg"):
        return False
    try:
        _unload_pipe()
        from diffusers import AudioLDM2Pipeline
        ap = AudioLDM2Pipeline.from_pretrained("cvssp/audioldm2", torch_dtype=torch.float16).to("cuda")
        try:
            gen = torch.Generator(device="cuda").manual_seed(seed)
            audio_len = max(2.5, min(10.0, float(duration_sec)))
            wav = ap(
                prompt=f"Realistic synchronized cinematic foley sound effects and atmosphere: {prompt}",
                negative_prompt="low quality, harsh noise, distortion, clipping",
                num_inference_steps=25,
                audio_length_in_s=audio_len,
                generator=gen,
            ).audios[0]
        finally:
            del ap
            gc.collect()
            torch.cuda.empty_cache()

        wav_path = video_path + ".wav"
        arr = np.asarray(wav, dtype=np.float32).reshape(-1)
        peak = float(np.max(np.abs(arr))) if arr.size else 1.0
        if peak > 1.0:
            arr = arr / peak
        pcm = np.clip(arr * 32767.0, -32768, 32767).astype(np.int16)
        with wave.open(wav_path, "wb") as wf:
            wf.setnchannels(1)
            wf.setsampwidth(2)
            wf.setframerate(16000)
            wf.writeframes(pcm.tobytes())

        muxed_path = video_path + ".mux.mp4"
        proc = subprocess.run(
            [
                "ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
                "-i", video_path, "-i", wav_path,
                "-map", "0:v:0", "-map", "1:a:0",
                "-c:v", "copy", "-c:a", "aac", "-b:a", "192k",
                "-shortest", "-movflags", "+faststart", muxed_path,
            ],
            capture_output=True,
            text=True,
            timeout=60,
        )
        if proc.returncode == 0 and os.path.isfile(muxed_path):
            os.replace(muxed_path, video_path)
            os.remove(wav_path)
            return True
    except Exception as exc:
        print("Audio synthesis skipped:", exc)
    return False

class ImageRequest(BaseModel):
    prompt: str
    negative_prompt: str = "blurry, low quality, distorted, watermark"
    width: int = 1024
    height: int = 576
    steps: int = 20
    cfg_scale: float = 7.0
    seed: int = -1
    model: str = "sdxl-turbo"

class VideoRequest(BaseModel):
    prompt: str
    negative_prompt: str = "blurry, jittery, distorted, worst quality"
    width: int = 704
    height: int = 480
    duration_sec: int = 4
    steps: int = 8
    cfg_scale: float = 3.0
    seed: int = -1
    model: str = "ltx-0.9.7-distilled"
    mode: str = "t2v"
    start_frame_data_url: Optional[str] = None
    end_frame_data_url: Optional[str] = None
    generate_audio: bool = True

@app.get("/api/health")
def health():
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    total_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 1) if torch.cuda.is_available() else 0.0
    free_gb = round(torch.cuda.mem_get_info(0)[0] / (1024 ** 3), 1) if torch.cuda.is_available() else 0.0
    return {
        "ok": True,
        "gpu_name": gpu_name,
        "vram_total_gb": total_gb,
        "vram_free_gb": free_gb,
        "active_pipeline": STATE["active_pipe_key"],
        "models": ["sdxl-turbo", "sdxl-base", "ltx-0.9.7-distilled", "wan2.1-1.3b", "cogvideox-2b", "ltx-video"],
    }

@app.post("/api/image")
def generate_image(req: ImageRequest):
    actual_seed = int(time.time() * 1000) % (2**31) if req.seed < 0 else int(req.seed)
    with GPU_LOCK:
        pipe = _load_image_pipe(req.model)
        is_turbo = req.model == "sdxl-turbo"
        w = min(max(256, (req.width // 16) * 16), 768 if is_turbo else 1152)
        h = min(max(256, (req.height // 16) * 16), 768 if is_turbo else 1152)
        steps = min(max(1, req.steps), 4) if is_turbo else min(max(12, req.steps), 40)
        cfg = 0.0 if is_turbo else float(req.cfg_scale)
        gen = torch.Generator(device="cuda").manual_seed(actual_seed)
        img = pipe(
            prompt=req.prompt,
            negative_prompt=None if is_turbo else req.negative_prompt,
            width=w,
            height=h,
            num_inference_steps=steps,
            guidance_scale=cfg,
            generator=gen,
        ).images[0]
        fname = f"img_{int(time.time())}_{actual_seed}.png"
        fpath = os.path.join("outputs", fname)
        img.save(fpath)
        buf = io.BytesIO()
        img.save(buf, format="PNG")
        b64 = base64.b64encode(buf.getvalue()).decode("ascii")
        return {
            "ok": True,
            "seed": actual_seed,
            "width": w,
            "height": h,
            "output_path": f"/outputs/{fname}",
            "data_url": f"data:image/png;base64,{b64}",
        }

def _run_video_worker(job_id: str, req: VideoRequest):
    from diffusers.utils import export_to_video
    job = STATE["jobs"][job_id]
    try:
        actual_seed = int(time.time() * 1000) % (2**31) if req.seed < 0 else int(req.seed)
        has_start = bool(req.start_frame_data_url and req.start_frame_data_url.startswith("data:image/"))
        has_end = bool(req.end_frame_data_url and req.end_frame_data_url.startswith("data:image/"))
        mode_variant = "flf" if (has_start and has_end) else ("i2v" if (req.mode == "i2v" and has_start) else "t2v")
        model_key = req.model or "ltx-0.9.7-distilled"
        if mode_variant in ("i2v", "flf") and model_key not in ("ltx-0.9.7-distilled", "ltx-video"):
            model_key = "ltx-0.9.7-distilled"

        job["status"] = "processing"
        job["progress"] = 15
        job["phase"] = f"Loading {model_key} ({mode_variant.upper()}) on NVIDIA T4 16 GB..."

        with GPU_LOCK:
            pipe = _load_video_pipe(model_key, mode_variant)
            job["progress"] = 35
            job["phase"] = f"Diffusing real AI video ({model_key})..."
            gen = torch.Generator(device="cpu").manual_seed(actual_seed)

            if model_key in ("ltx-0.9.7-distilled", "ltx-video"):
                w = max(384, min(768, (req.width // 32) * 32))
                h = max(384, min(512, (req.height // 32) * 32))
                fps = 24
                num_frames = min(97, max(25, int(req.duration_sec * fps) + 1))
                num_frames = ((num_frames - 1) // 8) * 8 + 1
                is_distilled = model_key == "ltx-0.9.7-distilled"
                steps = 8 if is_distilled else min(max(16, req.steps), 35)
                cfg = 1.0 if is_distilled else max(2.5, min(7.0, req.cfg_scale))
                kwargs = dict(
                    prompt=req.prompt,
                    negative_prompt=req.negative_prompt,
                    width=w,
                    height=h,
                    num_frames=num_frames,
                    num_inference_steps=steps,
                    guidance_scale=cfg,
                    generator=gen,
                )
                if mode_variant == "flf" and req.start_frame_data_url and req.end_frame_data_url:
                    from diffusers.pipelines.ltx.pipeline_ltx_condition import LTXVideoCondition
                    kwargs["conditions"] = [
                        LTXVideoCondition(image=_decode_pil(req.start_frame_data_url, w, h), frame_index=0, strength=1.0),
                        LTXVideoCondition(image=_decode_pil(req.end_frame_data_url, w, h), frame_index=num_frames - 1, strength=1.0),
                    ]
                elif mode_variant == "i2v" and req.start_frame_data_url:
                    kwargs["image"] = _decode_pil(req.start_frame_data_url, w, h)
                frames = pipe(**kwargs).frames[0]
            elif model_key == "wan2.1-1.3b":
                fps = 16
                frames = pipe(
                    prompt=req.prompt,
                    negative_prompt=req.negative_prompt,
                    height=480,
                    width=832,
                    num_frames=min(65, max(17, int(req.duration_sec * fps) + 1)),
                    num_inference_steps=min(max(18, req.steps), 40),
                    guidance_scale=5.0,
                    generator=gen,
                ).frames[0]
            else:
                fps = 8
                num_frames = min(49, max(17, int(req.duration_sec * fps) + 1))
                frames = pipe(
                    prompt=req.prompt,
                    num_videos_per_prompt=1,
                    num_inference_steps=min(max(15, req.steps), 40),
                    num_frames=num_frames,
                    guidance_scale=float(req.cfg_scale),
                    generator=gen,
                ).frames[0]

            job["progress"] = 82
            job["phase"] = "Encoding H.264 MP4 clip..."
            fname = f"vid_{job_id}_{actual_seed}.mp4"
            fpath = os.path.join("outputs", fname)
            export_to_video(frames, fpath, fps=fps)

            has_audio = False
            if req.generate_audio:
                job["progress"] = 90
                job["phase"] = "Synthesizing synchronized AI foley audio (AudioLDM2 + FFmpeg)..."
                has_audio = _synthesize_and_mux_audio(fpath, req.prompt, len(frames) / float(fps), actual_seed)

            job["status"] = "completed"
            job["progress"] = 100
            job["phase"] = "Completed on Colab T4 16 GB"
            job["output_path"] = f"/outputs/{fname}"
            job["has_audio"] = has_audio
    except Exception as exc:
        job["status"] = "failed"
        job["error"] = str(exc)

@app.post("/api/video")
def queue_video(req: VideoRequest):
    job_id = f"job_{uuid.uuid4().hex[:12]}"
    STATE["jobs"][job_id] = {
        "job_id": job_id,
        "status": "queued",
        "progress": 5,
        "phase": "Queued for NVIDIA T4 16 GB GPU...",
        "output_path": None,
        "has_audio": False,
        "error": None,
    }
    threading.Thread(target=_run_video_worker, args=(job_id, req), daemon=True).start()
    return {"ok": True, "job_id": job_id, "status": "queued"}

@app.get("/api/job/{job_id}")
def get_job(job_id: str):
    if not re.fullmatch(r"[A-Za-z0-9_-]{4,64}", job_id) or job_id not in STATE["jobs"]:
        raise HTTPException(status_code=404, detail="Job not found")
    return STATE["jobs"][job_id]

@app.get("/outputs/{filename}")
def get_output(filename: str):
    if not re.fullmatch(r"[A-Za-z0-9._-]{3,128}", filename):
        raise HTTPException(status_code=400, detail="Invalid filename")
    fpath = os.path.join("outputs", filename)
    if not os.path.isfile(fpath):
        raise HTTPException(status_code=404, detail="Output file not found")
    return FileResponse(fpath)

def _start_api():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    config = uvicorn.Config(app, host="127.0.0.1", port=7860, log_level="warning")
    server = uvicorn.Server(config)
    loop.run_until_complete(server.serve())

if not globals().get("_FLOWSTUDIO_SERVER_STARTED"):
    threading.Thread(target=_start_api, daemon=True).start()
    _FLOWSTUDIO_SERVER_STARTED = True

api_ready = False
for _ in range(30):
    try:
        with urllib.request.urlopen("http://127.0.0.1:7860/api/health", timeout=2) as r:
            if r.status == 200:
                print("OmniLocal 1.2 T4 API health OK:", r.read().decode("utf-8"))
                api_ready = True
                break
    except Exception:
        time.sleep(0.5)

if not api_ready:
    raise RuntimeError("Local FastAPI server failed to start on port 7860.")

cf_bin = "/usr/local/bin/cloudflared"
if not os.path.exists(cf_bin):
    print("Downloading official cloudflared binary...")
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        cf_bin,
    )
    os.chmod(cf_bin, 0o755)

print("Starting free Cloudflare Quick Tunnel to your T4 GPU...")
proc = subprocess.Popen(
    [cf_bin, "tunnel", "--url", "http://127.0.0.1:7860", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

tunnel_url = None
recent_lines = []
for line in proc.stdout:
    recent_lines.append(line.strip())
    if len(recent_lines) > 20:
        recent_lines.pop(0)
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m and not tunnel_url:
        tunnel_url = m.group(0)
        print("=" * 72)
        print("OMNILOCAL 1.2 COLAB T4 GPU SERVER IS LIVE!")
        print("Paste this URL into https://flow-studio-pwa.vercel.app -> Engine Settings -> Test Connection:")
        print("\n  " + tunnel_url + "\n")
        print("Keep this cell running while generating real AI video + audio.")
        print("=" * 72)

if not tunnel_url:
    raise RuntimeError("Cloudflare tunnel exited without printing a URL:\n" + "\n".join(recent_lines))
